In [1]:
import os
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from tqdm.auto import tqdm


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

seed_everything(1234)
print("Device:", device)

Device: cuda


In [2]:
DATA_ROOT = "./data"
RESULTS_DIR = "./results"

N_TRAIN = 500
N_VAL = 100
N_TEST = 100

EPOCHS = 100
BATCH_SIZE = 4

LR = 1e-3
BETA1 = 0.9
BETA2 = 0.999
EPS = 1e-7

SEEDS = [1234, 1235, 1236]

print("MNIST H3 dense-only experiment")
print(f"Split: train={N_TRAIN}, val={N_VAL}, test={N_TEST}")
print(f"Epochs: {EPOCHS}, batch size: {BATCH_SIZE}")
print(f"Seeds: {SEEDS}")

MNIST H3 dense-only experiment
Split: train=500, val=100, test=100
Epochs: 100, batch size: 4
Seeds: [1234, 1235, 1236]


In [3]:
mnist_train_full = MNIST(root=DATA_ROOT, train=True, download=True)
mnist_test_full = MNIST(root=DATA_ROOT, train=False, download=True)

x_mnist_train_full = mnist_train_full.data.clone().float()
y_mnist_train_full = mnist_train_full.targets.clone().long()

x_mnist_test_full = mnist_test_full.data.clone().float()
y_mnist_test_full = mnist_test_full.targets.clone().long()

print("Train:", x_mnist_train_full.shape, y_mnist_train_full.shape)
print("Test :", x_mnist_test_full.shape, y_mnist_test_full.shape)
print("Pixel range:", float(x_mnist_train_full.min()), "to", float(x_mnist_train_full.max()))

Train: torch.Size([60000, 28, 28]) torch.Size([60000])
Test : torch.Size([10000, 28, 28]) torch.Size([10000])
Pixel range: 0.0 to 255.0


In [4]:
def sample_mnist_dense_split(
    x_train_all: torch.Tensor,
    y_train_all: torch.Tensor,
    x_test_all: torch.Tensor,
    y_test_all: torch.Tensor,
    n_train: int = N_TRAIN,
    n_val: int = N_VAL,
    n_test: int = N_TEST,
    seed: int = 0,
):
    rng = random.Random(seed)

    train_indices = torch.tensor(
        rng.sample(range(len(x_train_all)), n_train + n_val),
        dtype=torch.long,
    )
    test_indices = torch.tensor(
        rng.sample(range(len(x_test_all)), n_test),
        dtype=torch.long,
    )

    x_trainval = x_train_all[train_indices]
    y_trainval = y_train_all[train_indices]

    split = {
        "x_train": x_trainval[:n_train].clone(),
        "y_train": y_trainval[:n_train].clone(),
        "x_val": x_trainval[n_train:n_train + n_val].clone(),
        "y_val": y_trainval[n_train:n_train + n_val].clone(),
        "x_test": x_test_all[test_indices].clone(),
        "y_test": y_test_all[test_indices].clone(),
    }
    return split


def make_dense_loaders(split: dict, batch_size: int = BATCH_SIZE):
    pin = device.type == "cuda"

    train_loader = DataLoader(
        TensorDataset(split["x_train"], split["y_train"]),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=pin,
    )
    val_loader = DataLoader(
        TensorDataset(split["x_val"], split["y_val"]),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
    )
    test_loader = DataLoader(
        TensorDataset(split["x_test"], split["y_test"]),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
    )
    return train_loader, val_loader, test_loader


example_split = sample_mnist_dense_split(
    x_mnist_train_full,
    y_mnist_train_full,
    x_mnist_test_full,
    y_mnist_test_full,
    seed=SEEDS[0],
)

for key, value in example_split.items():
    print(key, tuple(value.shape))

x_train (500, 28, 28)
y_train (500,)
x_val (100, 28, 28)
y_val (100,)
x_test (100, 28, 28)
y_test (100,)


In [5]:
def init_like_keras(module: nn.Module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class H3DenseMNIST(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 1024, bias=True),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(1024, num_classes, bias=True),
        )

        self.apply(init_like_keras)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model_h3 = H3DenseMNIST().to(device)
print(model_h3)

H3DenseMNIST(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=1024, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=1024, out_features=10, bias=True)
  )
)


In [6]:
@torch.no_grad()
def evaluate_dense(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_seen += xb.size(0)

    return total_loss / total_seen, total_correct / total_seen


def train_one_h3_mnist_run(run_seed: int, epochs: int = EPOCHS):
    seed_everything(run_seed)

    split = sample_mnist_dense_split(
        x_mnist_train_full,
        y_mnist_train_full,
        x_mnist_test_full,
        y_mnist_test_full,
        seed=run_seed,
    )
    train_loader, val_loader, test_loader = make_dense_loaders(split, batch_size=BATCH_SIZE)

    model = H3DenseMNIST().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(BETA1, BETA2),
        eps=EPS,
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(dim=1) == yb).sum().item()
            running_seen += xb.size(0)

        train_loss = running_loss / running_seen
        train_acc = running_correct / running_seen
        val_loss, val_acc = evaluate_dense(model, val_loader, criterion)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

    test_loss, test_acc = evaluate_dense(model, test_loader, criterion)

    return {
        "seed": run_seed,
        "history": pd.DataFrame(history),
        "test_loss": test_loss,
        "test_acc": test_acc,
    }

In [7]:
def run_h3_mnist_seed(seed: int, tag: str = "mnist_h3_dense"):
    start = time.perf_counter()

    result = train_one_h3_mnist_run(run_seed=seed, epochs=EPOCHS)

    elapsed_seconds = time.perf_counter() - start

    runs_df = pd.DataFrame([
        {
            "seed": result["seed"],
            "test_loss": result["test_loss"],
            "test_acc": result["test_acc"],
            "final_train_acc": result["history"]["train_acc"].iloc[-1],
            "final_val_acc": result["history"]["val_acc"].iloc[-1],
        }
    ])
    history_df = result["history"].copy()

    os.makedirs(RESULTS_DIR, exist_ok=True)

    runs_path = f"{RESULTS_DIR}/{tag}_seed{seed}_run.csv"
    hist_path = f"{RESULTS_DIR}/{tag}_seed{seed}_history.csv"

    runs_df.to_csv(runs_path, index=False)
    history_df.to_csv(hist_path, index=False)

    print(f"\nSeed: {seed}")
    print(f"Test accuracy : {result['test_acc']:.4f}")
    print(f"Test loss     : {result['test_loss']:.4f}")
    print(f"Final train   : {result['history']['train_acc'].iloc[-1]:.4f}")
    print(f"Final val     : {result['history']['val_acc'].iloc[-1]:.4f}")
    print(f"Elapsed time  : {elapsed_seconds / 60:.2f} minutes")
    print("\nSaved:")
    print(runs_path)
    print(hist_path)

    return runs_df, history_df

In [8]:
runs_seed1, history_seed1 = run_h3_mnist_seed(SEEDS[0])


Seed: 1234
Test accuracy : 0.9000
Test loss     : 93.4863
Final train   : 0.9960
Final val     : 0.8700
Elapsed time  : 0.68 minutes

Saved:
./results/mnist_h3_dense_seed1234_run.csv
./results/mnist_h3_dense_seed1234_history.csv


In [9]:
runs_seed2, history_seed2 = run_h3_mnist_seed(SEEDS[1])


Seed: 1235
Test accuracy : 0.8500
Test loss     : 192.0041
Final train   : 0.9860
Final val     : 0.8400
Elapsed time  : 0.73 minutes

Saved:
./results/mnist_h3_dense_seed1235_run.csv
./results/mnist_h3_dense_seed1235_history.csv


In [10]:
runs_seed3, history_seed3 = run_h3_mnist_seed(SEEDS[2])


Seed: 1236
Test accuracy : 0.8600
Test loss     : 100.7127
Final train   : 0.9860
Final val     : 0.8400
Elapsed time  : 0.70 minutes

Saved:
./results/mnist_h3_dense_seed1236_run.csv
./results/mnist_h3_dense_seed1236_history.csv


In [11]:
summary_runs = pd.concat(
    [runs_seed1, runs_seed2, runs_seed3],
    ignore_index=True
)

summary_table = summary_runs.copy()
summary_table.loc["mean"] = {
    "seed": "mean",
    "test_loss": summary_runs["test_loss"].mean(),
    "test_acc": summary_runs["test_acc"].mean(),
    "final_train_acc": summary_runs["final_train_acc"].mean(),
    "final_val_acc": summary_runs["final_val_acc"].mean(),
}

summary_table.loc["std"] = {
    "seed": "std",
    "test_loss": summary_runs["test_loss"].std(ddof=1),
    "test_acc": summary_runs["test_acc"].std(ddof=1),
    "final_train_acc": summary_runs["final_train_acc"].std(ddof=1),
    "final_val_acc": summary_runs["final_val_acc"].std(ddof=1),
}

summary_table.to_csv(f"{RESULTS_DIR}/mnist_h3_dense_3seed_summary.csv", index=False)

summary_table

,seed,test_loss,test_acc,final_train_acc,final_val_acc
0,1234,93.486315,0.900000,0.996000,0.870000
1,1235,192.004130,0.850000,0.986000,0.840000
2,1236,100.712748,0.860000,0.986000,0.840000
mean,mean,128.734398,0.870000,0.989333,0.850000
std,std,54.912199,0.026458,0.005774,0.017321
